# CS:GO Raunt Sonucu Tahmini

## Kütüphaneler ve Veri Yükleme

In [38]:
import zipfile
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

with zipfile.ZipFile('csdata.zip', 'r') as z:
    with z.open('mm_master_demos.csv') as f:
        df = pd.read_csv(f)

## Veri Hazırlama ve Özellik Mühendisliği

In [39]:
# Oyuncu/hasar bazlı tekrar eden satırları raunt bazına indirge
df_temiz = df[['file', 'round', 'map', 'winner_side', 'ct_eq_val', 't_eq_val']]
df_rauntlar = df_temiz.drop_duplicates(subset=['file', 'round']).copy()

# 1-15 arası Takım A T'dedir, 16 ve sonrasında Takım B T'ye geçer
df_rauntlar['t_team'] = np.where(df_rauntlar['round'] <= 15, 'Takim_A', 'Takim_B')
df_rauntlar['ct_team'] = np.where(df_rauntlar['round'] <= 15, 'Takim_B', 'Takim_A')

df_rauntlar['kazanan_takim'] = np.where(df_rauntlar['winner_side'] == 'Terrorist', df_rauntlar['t_team'], df_rauntlar['ct_team'])

In [40]:
df_rauntlar['a_win'] = (df_rauntlar['kazanan_takim'] == 'Takim_A').astype(int)
df_rauntlar['b_win'] = (df_rauntlar['kazanan_takim'] == 'Takim_B').astype(int)

# Takım A'nın mevcut raunttaki ekipman değeri farkı (Ekonomi avantajı)
df_rauntlar['a_eq_diff'] = np.where(
    df_rauntlar['round'] <= 15,
    df_rauntlar['t_eq_val'] - df_rauntlar['ct_eq_val'],
    df_rauntlar['ct_eq_val'] - df_rauntlar['t_eq_val']
)

df_rauntlar['a_is_t'] = np.where(df_rauntlar['t_team'] == 'Takim_A', 1, 0)
df_rauntlar['map_side'] = df_rauntlar['map'] + '_' + np.where(df_rauntlar['a_is_t'] == 1, 'T', 'CT')

In [41]:
def seri_hesapla(seri):
    sayac = 0
    lose_streak = []
    for i in seri:
        if i == 0:
            sayac += 1
        else:
            sayac = 0
        lose_streak.append(sayac)
    return lose_streak

df_rauntlar['a_losing_streak'] = df_rauntlar.groupby('file')['a_win'].transform(seri_hesapla)
df_rauntlar['b_losing_streak'] = df_rauntlar.groupby('file')['b_win'].transform(seri_hesapla)

# Veri sızıntısını önlemek için mevcut raundun değil, önceki raundun serisini al
df_rauntlar['a_prev_streak'] = df_rauntlar.groupby('file')['a_losing_streak'].shift(1).fillna(0)
df_rauntlar['b_prev_streak'] = df_rauntlar.groupby('file')['b_losing_streak'].shift(1).fillna(0)

## Model Eğitimi ve Değerlendirme

In [42]:
model_sutunlar = ['a_eq_diff', 'a_prev_streak', 'b_prev_streak', 'map_side', 'a_win']
df_model = df_rauntlar[model_sutunlar].dropna().copy()
gruplar = df_rauntlar.loc[df_model.index, 'file']  

df_model = pd.get_dummies(df_model, drop_first=True)

X = df_model.drop(columns=['a_win'])
y = df_model['a_win']

# Rastgele raunt yerine maç bazında ayırma
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=10)
train_idx, test_idx = next(gss.split(X, y, groups=gruplar))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

In [43]:
# Scaler sadece train setinden öğrenir
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train, y_train);



In [44]:
cols = list(X.columns)
idx_eq = cols.index("a_eq_diff")
idx_a_st = cols.index("a_prev_streak")
idx_b_st = cols.index("b_prev_streak")

cogunluk_acc = y_test.value_counts(normalize=True).max()

# Sadece ekipman farkı (a_eq_diff)
lr_eq = LogisticRegression().fit(X_train[:, [idx_eq]], y_train)
eq_acc = accuracy_score(y_test, lr_eq.predict(X_test[:, [idx_eq]]))
eq_auc = roc_auc_score(y_test, lr_eq.predict_proba(X_test[:, [idx_eq]])[:, 1])

# Sadece kaybetme serisi (a_prev_streak, b_prev_streak)
lr_st = LogisticRegression().fit(X_train[:, [idx_a_st, idx_b_st]], y_train)
st_acc = accuracy_score(y_test, lr_st.predict(X_test[:, [idx_a_st, idx_b_st]]))
st_auc = roc_auc_score(y_test, lr_st.predict_proba(X_test[:, [idx_a_st, idx_b_st]])[:, 1])

# Korelasyonlar
corr_a = np.corrcoef(X_train[:, idx_a_st], X_train[:, idx_eq])[0, 1]
corr_b = np.corrcoef(X_train[:, idx_b_st], X_train[:, idx_eq])[0, 1]
               
print(f"Baseline                         : Accuracy = %{cogunluk_acc*100:.2f}")
print(f"Sadece Ardışık Mağlubiyet Serisi : Accuracy = %{st_acc*100:.2f} | ROC-AUC = {st_auc:.4f}")
print(f"Sadece Ekipman Farkı             : Accuracy = %{eq_acc*100:.2f} | ROC-AUC = {eq_auc:.4f}")
print(f"Korelasyonlar                    : a_streak={corr_a:.4f}, b_streak={corr_b:.4f}")

Baseline                         : Accuracy = %50.92
Sadece Ardışık Mağlubiyet Serisi : Accuracy = %64.64 | ROC-AUC = 0.6488
Sadece Ekipman Farkı             : Accuracy = %65.01 | ROC-AUC = 0.7099
Korelasyonlar                    : a_streak=-0.5362, b_streak=0.5464
